# next-moe: checkpoint 30M → ще 20M на нових UK/EN даних

**Спочатку Runtime → Change runtime type → L4.** Notebook автономний, код вбудований.
Модель/Engram/BPE беруться з вашого checkpoint; це continued pretraining, не chat SFT.
Новий corpus run має окремий каталог і не перезаписує checkpoint 30M.

80% web + 20% освітніх PDF за tokens, 50/50 UK/EN за sampled sequences.
UK: FineWeb2 `ukr_Cyrl`; EN: FineWeb-Edu `CC-MAIN-2025-26`; PDF: FinePDFs-Edu UK/EN.
Ревізії закріплені, corpus caps — **20M tokens/lang** (≈160MB int32 train files).
Старий Wikipedia validation залишається byte-identical; новий holdout окремий.

**Навчання за замовчуванням іде до кінця +20M**, без ліміту 1000 updates.
Завершення GPU runtime знищує локальний Adam checkpoint. Drive зберігає weights
≈0.55GiB, metadata і token files; після втрати runtime буде warm restart.
Підготовка даних також витрачає час GPU-сеансу; corpus cache повторно не збирається.


In [1]:
import sys, subprocess, json, math, time, gc, shutil, hashlib, os
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'datasets==3.6.0', 'tokenizers==0.21.1', 'matplotlib>=3.8,<4'])
import torch
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Оберіть GPU runtime; для L4: Runtime → Change runtime type.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory/2**30)
print('Native BF16:', torch.cuda.is_bf16_supported(including_emulation=False))


PyTorch: 2.11.0+cu130
GPU: NVIDIA L4
VRAM GiB: 22.0343017578125
Native BF16: True


## Параметри та checkpoint

BASE_RUN — frozen checkpoint-30m-frozen із попередньої інструкції. Якщо його немає,
вкажіть каталог робочого checkpoint 30M. Він має містити weights.pt і data-v2/.
RUN_NAME нового етапу інший. COMPUTE_UNITS_PER_HOUR введіть із поточної панелі
Colab для **L4**: не використовуємо стару ціну G4 8.9 як тариф L4.


In [2]:
USE_DRIVE = True
STORAGE_PATH = '/content/drive/MyDrive/next-moe'
BASE_RUN_NAME = 'checkpoint-30m-frozen'
BASE_DATA_NAME = 'data-v2'
RUN_NAME = 'next-moe-30m-plus20m-webpdf-v1'
ADDITIONAL_TOKENS = 20_000_000
TOKENS_PER_LANGUAGE = 20_000_000
NEW_VALIDATION_DOCUMENTS = 128
PDF_FRACTION = .2
SEQ_LEN = 512
MICROBATCH = 4
GRAD_ACCUM = 1
PRECISION = 'auto_master'     # L4 supports native BF16; FP32 master Adam
LEARNING_RATE = 1e-4          # new phase, conservative LR compared with original 2e-4
RESTART_WARMUP = 100
RUN_EFFICIENCY_BENCHMARK = True
COMPUTE_UNITS_PER_HOUR = 1.54 # enter actual L4 units/hour; None measures speed only
SESSION_UPDATES = None       # None = train to +20M goal; 1000 = deliberately pause
RESUME_MODE = 'auto'         # auto / warm (explicit optimizer reset)
PERSISTENT_BUDGET_GIB = 4.5   # NEW run folder only, not entire Drive quota
SEED = 1234
SHUFFLE_BUFFER = 256
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STORAGE = Path(STORAGE_PATH)
BASE_RUN = STORAGE / BASE_RUN_NAME
OLD_DATA = BASE_RUN / BASE_DATA_NAME
RUN_DIR = STORAGE / RUN_NAME
LOCAL_DIR = Path('/content/next-moe-local') / RUN_NAME
CODE_DIR = Path('/content/next-moe-continuation-code')
DATA_DIR = RUN_DIR / 'data-stage3'
CACHE_DIR = LOCAL_DIR / 'data'
for folder in (RUN_DIR, LOCAL_DIR, CODE_DIR): folder.mkdir(parents=True, exist_ok=True)
if RUN_DIR.resolve() == BASE_RUN.resolve():
    raise ValueError('New run must not overwrite frozen base run.')


Mounted at /content/drive


## Вбудовані вихідні файли


In [3]:
SOURCES = {'next_moe/__init__.py': '"""Experimental, trainable next-moe reference implementation."""\nfrom .model import ModelConfig, NextMoE\n', 'next_moe/corpus.py': '"""Bounded, pinned UK/EN continuation corpus with unchanged BPE and holdouts."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport shutil\nimport numpy as np\nfrom tokenizers import Tokenizer\n\nSOURCES = {\n    \'web_uk\': {\'dataset\': \'HuggingFaceFW/fineweb-2\', \'config\': \'ukr_Cyrl\',\n               \'revision\': \'af9c13333eb981300149d5ca60a8e9d659b276b9\', \'license\': \'ODC-BY-1.0\'},\n    \'web_en\': {\'dataset\': \'HuggingFaceFW/fineweb-edu\', \'config\': \'CC-MAIN-2025-26\',\n               \'revision\': \'87f09149ef4734204d70ed1d046ddc9ca3f2b8f9\', \'license\': \'ODC-BY-1.0\'},\n    \'pdf_uk\': {\'dataset\': \'HuggingFaceFW/finepdfs-edu\', \'config\': \'ukr_Cyrl\',\n               \'revision\': \'9cfabe2127faca99b3d5c4dc6d1fcb397399ebde\', \'license\': \'ODC-BY-1.0\'},\n    \'pdf_en\': {\'dataset\': \'HuggingFaceFW/finepdfs-edu\', \'config\': \'eng_Latn\',\n               \'revision\': \'9cfabe2127faca99b3d5c4dc6d1fcb397399ebde\', \'license\': \'ODC-BY-1.0\'},\n}\n\n\ndef digest(text):\n    return hashlib.sha256(text.encode(\'utf-8\')).hexdigest()\n\n\nclass NearDuplicates:\n    """Approximate SimHash on bounded word shingles; not a proof of no leakage."""\n    def __init__(self):\n        self.bands = defaultdict(list)\n\n    @staticmethod\n    def fingerprint(text):\n        words = re.findall(r\'\\w+\', text.casefold())\n        shingles = {\' \'.join(words[i:i+5]) for i in range(max(len(words)-4, 1))}\n        # Deterministic bounded sample; long texts cannot exhaust host RAM/CPU here.\n        hashes = sorted(hashlib.blake2b(s.encode(), digest_size=8).digest() for s in shingles)[:128]\n        bits = np.unpackbits(np.frombuffer(b\'\'.join(hashes), dtype=np.uint8)).reshape(-1, 64)\n        packed = np.packbits(bits.sum(axis=0)*2 >= len(hashes)).tobytes()\n        return int.from_bytes(packed, \'big\')\n\n    def duplicate(self, text, add=False):\n        value = self.fingerprint(text)\n        keys = [(i, (value >> (i*16)) & 65535) for i in range(4)]\n        candidates = {v for key in keys for v in self.bands[key]}\n        match = any((value ^ other).bit_count() <= 3 for other in candidates)\n        if add and not match:\n            for key in keys:\n                self.bands[key].append(value)\n        return match\n\n\ndef stream_source(spec, seed, shuffle_buffer):\n    from datasets import load_dataset\n    print(\'Opening\', spec[\'dataset\'], spec[\'config\'], spec[\'revision\'], flush=True)\n    dataset = load_dataset(spec[\'dataset\'], name=spec[\'config\'], revision=spec[\'revision\'],\n                           split=\'train\', streaming=True)\n    return iter(dataset.shuffle(seed=seed, buffer_size=shuffle_buffer))\n\n\ndef prepare_continuation_data(output, old_data, tokens_per_language=20_000_000,\n                              validation_documents=128, pdf_fraction=.2, seed=1234,\n                              shuffle_buffer=256, max_chars=20000, max_scanned=500000,\n                              record_sources=None, progress_every=500):\n    """80/20 web/PDF token pool, 50/50 languages in Trainer; no tokenizer training.\n\n    Old val bins are byte-identical; new domain holdouts are separate. All source\n    readers fail explicitly, never fall back to toy data or old Wikipedia train.\n    record_sources is for offline fixtures only, keyed by web_uk/pdf_uk/etc.\n    """\n    if tokens_per_language < 1 or validation_documents < 2 or not 0 < pdf_fraction < 1:\n        raise ValueError(\'Positive corpus limits and pdf_fraction strictly between 0 and 1 required.\')\n    output, old_data = Path(output), Path(old_data)\n    recipe = {\'tokens_per_language\': tokens_per_language, \'validation_documents\': validation_documents,\n              \'pdf_fraction\': pdf_fraction, \'seed\': seed, \'shuffle_buffer\': shuffle_buffer,\n              \'max_chars\': max_chars, \'max_scanned\': max_scanned, \'sources\': SOURCES,\n              \'old_tokenizer_sha256\': hashlib.sha256((old_data/\'tokenizer.json\').read_bytes()).hexdigest(),\n              \'old_metadata_sha256\': hashlib.sha256((old_data/\'metadata.json\').read_bytes()).hexdigest(),\n              \'old_validation_sha256\': {lang: hashlib.sha256((old_data/f\'val_{lang}.bin\').read_bytes()).hexdigest()\n                                        for lang in (\'uk\', \'en\')}}\n    if (output/\'metadata.json\').exists():\n        meta = json.loads((output/\'metadata.json\').read_text())\n        if meta.get(\'recipe\') != recipe:\n            raise ValueError(\'Existing corpus recipe differs. Use a NEW output directory.\')\n        for name, expected in meta[\'file_sha256\'].items():\n            if hashlib.sha256((output/name).read_bytes()).hexdigest() != expected:\n                raise ValueError(f\'Cached corpus file changed: {name}\')\n        print(\'Verified cached corpus:\', output, flush=True)\n        return meta\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(\'Incomplete corpus directory exists. Use a new directory or remove the incomplete copy.\')\n    output.mkdir(parents=True, exist_ok=True)\n    tok = Tokenizer.from_file(str(old_data/\'tokenizer.json\'))\n    eos = tok.token_to_id(\'<eos>\')\n    old = json.loads((old_data/\'metadata.json\').read_text())\n    blocked, near = set(), NearDuplicates()\n    for lang in (\'uk\', \'en\'):\n        shutil.copy2(old_data/f\'val_{lang}.bin\', output/f\'val_{lang}.bin\')\n        origins_path = old_data/\'documents.json\'\n        if origins_path.exists():\n            origins = json.loads(origins_path.read_text())[lang][\'val\']\n            for origin in origins:\n                blocked.update(origin[k] for k in (\'sha256_used_text\', \'sha256_full_stripped_text\') if origin.get(k))\n        blocked.update(old.get(\'document_hashes\', {}).get(lang, {}).get(\'val\', []))\n        ids = np.fromfile(old_data/f\'val_{lang}.bin\', dtype=np.int32)\n        start = 0\n        for stop in np.flatnonzero(ids == eos):\n            text = tok.decode(ids[start:stop].tolist(), skip_special_tokens=False).strip()\n            if text:\n                blocked.add(digest(text)); near.duplicate(text, add=True)\n            start = stop+1\n    shutil.copy2(old_data/\'tokenizer.json\', output/\'tokenizer.json\')\n    counts, scanned, rejected, doc_counts, source_counts = {}, {}, {}, {}, {}\n    hashes = {lang: {\'val\': old[\'document_hashes\'][lang][\'val\'], \'train\': [], \'val_new\': []}\n              for lang in (\'uk\', \'en\')}\n    with (output/\'documents.jsonl\').open(\'w\') as manifest:\n        for lang in (\'uk\', \'en\'):\n            specs = {kind: SOURCES[f\'{kind}_{lang}\'] for kind in (\'web\', \'pdf\')}\n            readers = {kind: (iter(record_sources[f\'{kind}_{lang}\']) if record_sources is not None\n                              else stream_source(spec, seed, shuffle_buffer)) for kind, spec in specs.items()}\n            train_counts, val_counts, scanned_lang = Counter(), Counter(), 0\n            target = {\'web\': round(tokens_per_language*(1-pdf_fraction)),\n                      \'pdf\': tokens_per_language-round(tokens_per_language*(1-pdf_fraction))}\n            reject = Counter()\n            with (output/f\'train_{lang}.bin\').open(\'wb\') as train, (output/f\'val_new_{lang}.bin\').open(\'wb\') as val:\n                while any(train_counts[k] < target[k] for k in target):\n                    kind = min((k for k in target if train_counts[k] < target[k]),\n                               key=lambda k: train_counts[k]/max(target[k], 1))\n                    try:\n                        record = next(readers[kind])\n                    except StopIteration as error:\n                        raise ValueError(f\'{lang}/{kind} exhausted before token goal; do not silently change mixture.\') from error\n                    scanned_lang += 1\n                    if scanned_lang % progress_every == 0:\n                        print(lang, \'scanned\', scanned_lang, \'train tokens\', sum(train_counts.values()),\n                              \'/\', tokens_per_language, \'rejected\', dict(reject), flush=True)\n                    if scanned_lang > max_scanned:\n                        raise ValueError(f\'{lang} exceeded max_scanned before token goal.\')\n                    full = record.get(\'text\', \'\').strip()\n                    text = full[:max_chars]\n                    page_languages = record.get(\'per_page_languages\') or []\n                    known = [value for value in page_languages if value not in (\'unknown\', \'und\', None)]\n                    expected_language = \'ukr_Cyrl\' if lang == \'uk\' else \'eng_Latn\'\n                    if known and sum(value == expected_language for value in known)/len(known) < .8:\n                        reject[\'mixed_pdf_languages\'] += 1; continue\n                    if len(text) < 200 or sum(c.isalpha() for c in text)/max(len(text), 1) < .35:\n                        reject[\'short_or_nonprose\'] += 1; continue\n                    lines = [line.strip() for line in text.splitlines() if line.strip()]\n                    if len(lines) > 10 and len(set(lines))/len(lines) < .5:\n                        reject[\'repeated_lines\'] += 1; continue\n                    key, full_key = digest(text), digest(full)\n                    if key in blocked or full_key in blocked or near.duplicate(text):\n                        reject[\'duplicate_or_holdout_match\'] += 1; continue\n                    blocked.update((key, full_key)); near.duplicate(text, add=True)\n                    ids = np.asarray(tok.encode(text).ids + [eos], dtype=np.int32)\n                    # Holdouts per source ensure both web and PDF coverage.\n                    pdf_val = min(validation_documents-1, max(1, round(validation_documents*pdf_fraction)))\n                    val_goal = pdf_val if kind == \'pdf\' else validation_documents-pdf_val\n                    split = \'val_new\' if val_counts[kind] < val_goal else \'train\'\n                    ids.tofile(val if split == \'val_new\' else train)\n                    if split == \'val_new\':\n                        val_counts[kind] += 1\n                    else:\n                        train_counts[kind] += len(ids)\n                    hashes[lang][split].append(key)\n                    manifest.write(json.dumps({\'language\': lang, \'split\': split, \'source\': kind, **specs[kind],\n                        \'id\': record.get(\'id\'), \'url\': record.get(\'url\'),\n                        \'crawl_date\': record.get(\'date\') or record.get(\'crawl_date\'),\n                        \'sha256_used_text\': key, \'sha256_full_stripped_text\': full_key,\n                        \'tokens\': len(ids), \'used_characters\': len(text)}, ensure_ascii=False)+\'\\n\')\n            counts[f\'train_{lang}\'] = sum(train_counts.values())\n            counts[f\'val_{lang}\'] = (output/f\'val_{lang}.bin\').stat().st_size//4\n            counts[f\'val_new_{lang}\'] = (output/f\'val_new_{lang}.bin\').stat().st_size//4\n            doc_counts[lang] = {\'train\': len(hashes[lang][\'train\']), \'val_new\': sum(val_counts.values()),\n                                \'val\': len(hashes[lang][\'val\'])}\n            source_counts[lang], scanned[lang], rejected[lang] = dict(train_counts), scanned_lang, dict(reject)\n            print(lang, \'COMPLETE:\', counts[f\'train_{lang}\'], \'train tokens\', flush=True)\n    files = [\'tokenizer.json\', \'documents.jsonl\'] + [f\'{split}_{lang}.bin\'\n                 for lang in (\'uk\', \'en\') for split in (\'train\', \'val\', \'val_new\')]\n    meta = {\'source\': \'fineweb-finepdfs-continuation\', \'source_revision\': SOURCES, \'recipe\': recipe,\n            \'actual_vocab\': tok.get_vocab_size(), \'special_ids\': {s: tok.token_to_id(s) for s in (\'<pad>\', \'<eos>\')},\n            \'tokens\': counts, \'document_hashes\': hashes, \'document_counts\': doc_counts,\n            \'source_training_tokens\': source_counts, \'scanned_documents\': scanned, \'rejected\': rejected,\n            \'tokenizer_reused\': True, \'validation_preserved\': True,\n            \'filters\': \'min200 chars; alpha>=35%; known PDF page languages>=80% target; repeated-line filter; global exact hashes and approximate SimHash<=3\',\n            \'max_chars_per_document\': max_chars,\n            \'file_sha256\': {name: hashlib.sha256((output/name).read_bytes()).hexdigest() for name in files}}\n    (output/\'metadata.json\').write_text(json.dumps(meta, ensure_ascii=False, indent=2))\n    return meta\n\n\ndef install_domain_evaluation(trainer):\n    """Report old Wikipedia and new-domain holdouts without modifying core trainer."""\n    if getattr(trainer, \'_domain_evaluation_installed\', False):\n        return\n    original = trainer.evaluate\n    newer = {(\'val\', lang): np.memmap(trainer.data_dir/f\'val_new_{lang}.bin\', mode=\'r\', dtype=np.int32)\n             for lang in (\'uk\', \'en\')}\n    if any(len(a) <= trainer.cfg.sequence_length for a in newer.values()):\n        raise ValueError(\'New holdouts are too short for evaluation.\')\n    def evaluate():\n        result = original()\n        previous = {key: trainer.batches.arrays[key] for key in newer}\n        trainer.batches.arrays.update(newer)\n        try:\n            result.update({key.replace(\'val_\', \'val_new_\', 1): value for key, value in original().items()})\n        finally:\n            trainer.batches.arrays.update(previous)\n        return result\n    trainer.evaluate = evaluate\n    trainer._domain_evaluation_installed = True\n\n\ndef install_phase_schedule(trainer, initial_tokens, target_tokens):\n    """Cosine tracks THIS phase\'s token budget, independent of historical steps."""\n    if target_tokens <= initial_tokens:\n        raise ValueError(\'Phase target must exceed starting tokens.\')\n    def learning_rate():\n        progress = min(1., max(0., (trainer.tokens_seen-initial_tokens)/(target_tokens-initial_tokens)))\n        rate = trainer.cfg.learning_rate * (.1 + .9*.5*(1+math.cos(math.pi*progress)))\n        if trainer.restart_step is not None and trainer.cfg.restart_warmup > 0:\n            rate *= min(1., (trainer.step-trainer.restart_step+1)/trainer.cfg.restart_warmup)\n        return rate\n    trainer.learning_rate = learning_rate\n', 'next_moe/data.py': '"""Bounded UK/EN document ingestion, leak-free split, tokenizer and token files."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport numpy as np\nfrom tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders\n\nLANGUAGES = (\'uk\', \'en\')\nSPECIAL = [\'<pad>\', \'<bos>\', \'<eos>\', \'<unk>\']\nWIKIPEDIA_REVISION = \'b04c8d1ceb2f5cd4588862100d08de323dccfbaa\'\n\n\ndef document_sources(source=\'wikipedia\', local_dir=None):\n    if source == \'wikipedia\':\n        from datasets import load_dataset\n        # Public parquet dataset. No trust_remote_code; no full-dataset download.\n        return {lang: iter(load_dataset(\n            \'wikimedia/wikipedia\', f\'20231101.{lang}\', revision=WIKIPEDIA_REVISION,\n            split=\'train\', streaming=True))\n                for lang in LANGUAGES}\n    if source != \'local\' or local_dir is None:\n        raise ValueError(\'Use wikipedia or local with a directory containing uk/ and en/.\')\n    result = {}\n    for lang in LANGUAGES:\n        paths = sorted((Path(local_dir) / lang).glob(\'*.txt\'))\n        if len(paths) < 2:\n            raise ValueError(f\'Need at least two separate UTF-8 documents in {lang}/.\')\n        result[lang] = ({\'text\': path.read_text(encoding=\'utf-8\'), \'id\': path.name,\n                         \'url\': None, \'title\': path.name} for path in paths)\n    return result\n\n\ndef prepare_data(output, source=\'wikipedia\', local_dir=None, documents_per_language=1000,\n                 validation_documents=32, vocab_size=65536, max_chars_per_document=20000,\n                 seed=1234, tokenizer_path=None, fixed_validation_metadata=None,\n                 max_characters_per_language=None):\n    if documents_per_language <= validation_documents or validation_documents < 1:\n        raise ValueError(\'Training and validation must both contain documents.\')\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    old = (json.loads(Path(fixed_validation_metadata).read_text())\n           if fixed_validation_metadata is not None else None)\n    if old and tokenizer_path is None:\n        raise ValueError(\'Preserving old validation requires its original tokenizer.\')\n    reserved_owner = ({digest: lang for lang in LANGUAGES for digest in old[\'document_hashes\'][lang][\'val\']}\n                      if old else {})\n    raw = document_sources(source, local_dir)\n    documents, seen, hashes, provenance, scanned, used_characters = {}, set(), {}, {}, {}, {}\n    for lang in LANGUAGES:\n        reserved = set(old[\'document_hashes\'][lang][\'val\']) if old else set()\n        docs, ids, origins = [], [], []\n        scanned[lang], used_characters[lang] = 0, 0\n        for record in raw[lang]:\n            scanned[lang] += 1\n            text = record[\'text\'].strip()\n            if len(text) < 200:\n                continue\n            full_digest = hashlib.sha256(text.encode(\'utf-8\')).hexdigest()\n            text = text[:max_chars_per_document]\n            digest = hashlib.sha256(text.encode(\'utf-8\')).hexdigest()\n            if digest in seen or (digest in reserved_owner and reserved_owner[digest] != lang):\n                continue\n            if max_characters_per_language is not None and used_characters[lang] + len(text) > max_characters_per_language:\n                break\n            seen.add(digest)\n            docs.append(text)\n            ids.append(digest)\n            used_characters[lang] += len(text)\n            origins.append({\'id\': record.get(\'id\'), \'url\': record.get(\'url\'),\n                            \'title\': record.get(\'title\'), \'sha256_used_text\': digest,\n                            \'sha256_full_stripped_text\': full_digest, \'used_characters\': len(text)})\n            if len(docs) >= documents_per_language and reserved.issubset(set(ids)):\n                break\n        if old:\n            if not reserved.issubset(set(ids)):\n                raise ValueError(f\'Cannot recover all original {lang} validation documents; \'\n                                 \'increase corpus limits and keep source/truncation unchanged.\')\n            val = [ids.index(h) for h in old[\'document_hashes\'][lang][\'val\']]\n            train = [i for i, h in enumerate(ids) if h not in reserved]\n        else:\n            if len(docs) < validation_documents + 1:\n                raise ValueError(f\'Too few usable {lang} documents ({len(docs)}).\')\n            order = np.random.default_rng(seed).permutation(len(docs))\n            val, train = list(order[:validation_documents]), list(order[validation_documents:])\n        if not train:\n            raise ValueError(f\'No training documents for {lang}.\')\n        documents[lang], hashes[lang], provenance[lang] = {}, {}, {}\n        for split, indices in [(\'val\', val), (\'train\', train)]:\n            documents[lang][split] = [docs[i] for i in indices]\n            hashes[lang][split] = [ids[i] for i in indices]\n            provenance[lang][split] = [origins[i] for i in indices]\n    if tokenizer_path is not None:\n        tokenizer = Tokenizer.from_file(str(tokenizer_path))\n        if tokenizer.get_vocab_size() > vocab_size:\n            raise ValueError(\'Existing tokenizer exceeds model vocabulary.\')\n        import shutil\n        shutil.copy2(tokenizer_path, output / \'tokenizer.json\')  # exact bytes, unchanged IDs\n    else:\n        tokenizer = Tokenizer(models.BPE(unk_token=\'<unk>\'))\n        tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)\n        tokenizer.decoder = decoders.ByteLevel()\n        trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2, special_tokens=SPECIAL,\n                                      initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\n        tokenizer.train_from_iterator((doc for lang in LANGUAGES for doc in documents[lang][\'train\']),\n                                      trainer=trainer)\n        tokenizer.save(str(output / \'tokenizer.json\'))\n    metadata = {\'source\': source, \'source_config\': \'20231101.{uk,en}\' if source == \'wikipedia\' else None,\n                \'source_dataset\': \'wikimedia/wikipedia\' if source == \'wikipedia\' else None,\n                \'source_revision\': WIKIPEDIA_REVISION if source == \'wikipedia\' else None,\n                \'requested_documents_per_language\': documents_per_language,\n                \'requested_validation_documents_per_language\': validation_documents,\n                \'validation_documents_per_language\': len(documents[\'uk\'][\'val\']) if len(documents[\'uk\'][\'val\']) == len(documents[\'en\'][\'val\']) else None,\n                \'max_characters_per_document\': max_chars_per_document,\n                \'max_characters_per_language\': max_characters_per_language,\n                \'used_characters\': used_characters,\n                \'min_characters_per_document\': 200, \'scanned_documents\': scanned,\n                \'document_counts\': {lang: {split: len(documents[lang][split])\n                                          for split in (\'train\', \'val\')} for lang in LANGUAGES},\n                \'seed\': seed, \'document_hashes\': hashes, \'requested_vocab\': vocab_size,\n                \'actual_vocab\': tokenizer.get_vocab_size(), \'normalization\': \'none; byte-level BPE\',\n                \'tokenizer_reused\': tokenizer_path is not None,\n                \'validation_preserved\': old is not None,\n                \'special_ids\': {s: tokenizer.token_to_id(s) for s in SPECIAL}, \'tokens\': {}}\n    for lang in LANGUAGES:\n        for split in (\'train\', \'val\'):\n            count = 0\n            with (output / f\'{split}_{lang}.bin\').open(\'wb\') as handle:\n                for doc in documents[lang][split]:\n                    ids = tokenizer.encode(doc).ids + [tokenizer.token_to_id(\'<eos>\')]\n                    np.asarray(ids, dtype=np.int32).tofile(handle)\n                    count += len(ids)\n            metadata[\'tokens\'][f\'{split}_{lang}\'] = count\n    (output / \'documents.json\').write_text(json.dumps(provenance, ensure_ascii=False, indent=2))\n    (output / \'metadata.json\').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))\n    print(json.dumps({key: metadata[key] for key in (\'document_counts\', \'actual_vocab\', \'tokens\', \'special_ids\')}, indent=2))\n    return metadata\n\n\nclass TokenBatches:\n    """50/50 UK/EN sampling, no padding; CPU RNG state is checkpointed."""\n    def __init__(self, folder, sequence_length=128, microbatch=1, seed=1234):\n        import torch\n        self.torch = torch\n        self.length, self.batch = sequence_length, microbatch\n        self.rng = torch.Generator().manual_seed(seed)\n        self.arrays = {(split, lang): np.memmap(Path(folder) / f\'{split}_{lang}.bin\',\n                                              mode=\'r\', dtype=np.int32)\n                       for split in (\'train\', \'val\') for lang in LANGUAGES}\n        for key, array in self.arrays.items():\n            if len(array) <= sequence_length:\n                raise ValueError(f\'{key} needs more than {sequence_length} tokens.\')\n\n    def get(self, split, device, language=None, generator=None):\n        torch = self.torch\n        gen = self.rng if generator is None else generator\n        xs, ys = [], []\n        for _ in range(self.batch):\n            lang = language or LANGUAGES[int(torch.randint(2, (1,), generator=gen))]\n            array = self.arrays[(split, lang)]\n            start = int(torch.randint(len(array) - self.length, (1,), generator=gen))\n            row = torch.from_numpy(np.array(array[start:start + self.length + 1], dtype=np.int64))\n            xs.append(row[:-1]); ys.append(row[1:])\n        return torch.stack(xs).to(device), torch.stack(ys).to(device)\n', 'next_moe/model.py': '"""Trainable v0.3 reference: GR + causal Engram + GDN/GQA + no-drop MoE.\n\nThe chunkwise GDN is an exact triangular reformulation of the recurrence,\nnot a production Triton kernel. Generation recomputes the prefix (no KV API).\n"""\nfrom dataclasses import asdict, dataclass, field\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass ModelConfig:\n    vocab_size: int = 65536\n    layers: int = 12\n    dim: int = 512\n    experts: int = 16\n    top_k: int = 2\n    expert_dim: int = 448\n    branches: int = 4\n    rank: int = 64\n    q_heads: int = 8\n    kv_heads: int = 2\n    head_dim: int = 64\n    gdn_k_heads: int = 4\n    gdn_v_heads: int = 8\n    gdn_key_dim: int = 64\n    gdn_value_dim: int = 64\n    table_rows: list[int] = field(default_factory=lambda: [390647, 390653, 390671, 390673,\n                                                          390703, 390707, 390721, 390727])\n    row_dim: int = 32\n    full_every: int = 4\n    kv_group: int = 2\n    memory_layer: int = 1  # zero-based: decoder block 2\n    gdn_chunk: int = 32\n    rope_theta: float = 10000.0\n    eos_id: int = 2\n    pad_id: int = 0\n    checkpoint_blocks: bool = True\n\n    def __post_init__(self):\n        if self.layers % self.full_every or not 0 <= self.memory_layer < self.layers:\n            raise ValueError(\'Layers must end with a full layer; memory layer must exist.\')\n        if self.q_heads % self.kv_heads or self.gdn_v_heads % self.gdn_k_heads:\n            raise ValueError(\'Head ratios must be integral.\')\n        if not 1 <= self.top_k <= self.experts or len(self.table_rows) != 8:\n            raise ValueError(\'Invalid expert or Engram configuration.\')\n        if self.kv_group != 2 or self.head_dim % 2 or self.gdn_chunk < 1:\n            raise ValueError(\'This reference supports KV pairs and even RoPE dimensions.\')\n\n    @classmethod\n    def tiny(cls):\n        """Only for correctness tests; not the 0.2B+0.1B profile."""\n        return cls(vocab_size=512, layers=8, dim=32, experts=4, top_k=2,\n                   expert_dim=24, rank=4, q_heads=4, kv_heads=2, head_dim=8,\n                   gdn_k_heads=2, gdn_v_heads=4, gdn_key_dim=8, gdn_value_dim=8,\n                   table_rows=[101, 103, 107, 109, 113, 127, 131, 137], row_dim=4,\n                   gdn_chunk=4)\n\n    def to_dict(self):\n        return asdict(self)\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, shape, eps=1e-6):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(shape))\n        self.eps = eps\n\n    def forward(self, x):\n        y = x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps)\n        return (y * self.weight.float()).to(x.dtype)\n\n\nclass GatedResidual(nn.Module):\n    def __init__(self, cfg, write=True):\n        super().__init__()\n        self.n, self.d = cfg.branches, cfg.dim\n        self.norm = RMSNorm((self.n, self.d))\n        self.down = nn.Linear(self.n * self.d, cfg.rank, bias=False)\n        self.up = nn.Linear(cfg.rank, self.n * self.d, bias=False)\n        self.write = nn.Linear(self.n * self.d, self.n, bias=False) if write else None\n\n    def read(self, r):\n        normalized = self.norm(r)\n        z = normalized.flatten(-2)\n        gate = self.up(F.silu(self.down(z) / self.n)).sigmoid().reshape_as(r)\n        x = (gate * normalized).mean(-2)\n        s = 2 * (self.write(z) / self.n).sigmoid() if self.write is not None else None\n        return x, s\n\n    @staticmethod\n    def add(r, y, s):\n        return r + s.unsqueeze(-1).to(r.dtype) * y.unsqueeze(-2).to(r.dtype)\n\n\nclass CausalConv(nn.Module):\n    def __init__(self, width, kernel=4, dilation=1):\n        super().__init__()\n        self.left = (kernel - 1) * dilation\n        self.conv = nn.Conv1d(width, width, kernel, groups=width,\n                              dilation=dilation, bias=False)\n\n    def forward(self, x):\n        return self.conv(F.pad(x.transpose(1, 2), (self.left, 0))).transpose(1, 2)\n\n\nclass Engram(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.tables = nn.ModuleList([nn.Embedding(rows, cfg.row_dim) for rows in cfg.table_rows])\n        m, n, d = 8 * cfg.row_dim, cfg.branches, cfg.dim\n        self.value = nn.Linear(m, d, bias=False)\n        self.key = nn.Linear(m, n * d, bias=False)\n        self.query_norm, self.key_norm, self.conv_norm = [RMSNorm((n, d)) for _ in range(3)]\n        self.conv = CausalConv(n * d, kernel=4, dilation=3)\n\n    def addresses(self, tokens):\n        # Token IDs are kept unchanged: no accent stripping or case folding.\n        # Reset suffix lookup at EOS; left padding is deterministic at sample starts.\n        lag1 = F.pad(tokens[:, :-1], (1, 0), value=self.cfg.pad_id)\n        lag2 = F.pad(tokens[:, :-2], (2, 0), value=self.cfg.pad_id)[:, :tokens.shape[1]]\n        reset = tokens.eq(self.cfg.eos_id) | lag1.eq(self.cfg.eos_id)\n        lag1 = torch.where(reset, self.cfg.pad_id, lag1)\n        lag2 = torch.where(reset | lag2.eq(self.cfg.eos_id), self.cfg.pad_id, lag2)\n        hashes = []\n        for i, rows in enumerate(self.cfg.table_rows):\n            # Small integer coefficients keep arithmetic far below int64 overflow.\n            code = (tokens + 1) * (1009 + 104 * i) + (lag1 + 1) * (9176 + 62 * i)\n            if i >= 4:\n                code = code + (lag2 + 1) * (65537 + 50 * i)\n            hashes.append(code.remainder(rows))\n        return hashes\n\n    def forward(self, tokens, r):\n        e = torch.cat([table(ids) for table, ids in zip(self.tables, self.addresses(tokens))], -1)\n        e = e.to(r.dtype)\n        v = self.value(e)\n        key = self.key(e).reshape_as(r)\n        score = (self.query_norm(r).float() * self.key_norm(key).float()).sum(-1) / math.sqrt(self.cfg.dim)\n        transformed = score.sign() * score.abs().clamp_min(1e-6).sqrt()\n        u = transformed.sigmoid().unsqueeze(-1).to(r.dtype) * v.unsqueeze(-2)\n        conv = self.conv(self.conv_norm(u).flatten(-2)).reshape_as(r)\n        return r + u.to(r.dtype) + F.silu(conv).to(r.dtype)\n\n\ndef delta_chunk(q, k, v, log_alpha, beta, state):\n    """Exact gated-delta recurrence within one chunk. Shapes B,H,T,D; state B,H,K,V.\n\n    Solve write residuals against a unit-lower-triangular matrix. FP32 avoids\n    accumulation loss; no attention across future positions is performed.\n    """\n    with torch.autocast(device_type=q.device.type, enabled=False):\n        q, k, v, log_alpha, beta, state = [x.float() for x in (q, k, v, log_alpha, beta, state)]\n        length = q.shape[-2]\n        cumulative = log_alpha.cumsum(-1)\n        decay = cumulative.exp()\n        ratios = (cumulative.unsqueeze(-1) - cumulative.unsqueeze(-2)).clamp_max(0).exp()\n        kk = k @ k.transpose(-1, -2)\n        lower = (ratios * kk * beta.unsqueeze(-2)).tril(-1)\n        matrix = lower + torch.eye(length, device=q.device, dtype=torch.float32)\n        rhs = v - decay.unsqueeze(-1) * (k @ state)\n        errors = torch.linalg.solve_triangular(matrix, rhs, upper=False, unitriangular=True)\n        weights = (ratios * (q @ k.transpose(-1, -2)) * beta.unsqueeze(-2)).tril()\n        y = decay.unsqueeze(-1) * (q @ state) + weights @ errors\n        last_ratios = (cumulative[..., -1:] - cumulative).exp()\n        next_state = decay[..., -1, None, None] * state + k.transpose(-1, -2) @ (\n            errors * (last_ratios * beta).unsqueeze(-1))\n        return y, next_state\n\n\nclass GDN(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        k = cfg.gdn_k_heads * cfg.gdn_key_dim\n        w = cfg.gdn_v_heads * cfg.gdn_value_dim\n        self.qkv = nn.Linear(cfg.dim, 2 * k + w, bias=False)\n        self.conv = CausalConv(2 * k + w)\n        self.decay = nn.Linear(cfg.dim, cfg.gdn_v_heads, bias=False)\n        self.beta = nn.Linear(cfg.dim, cfg.gdn_v_heads, bias=False)\n        self.gate = nn.Linear(cfg.dim, w, bias=False)\n        self.out = nn.Linear(w, cfg.dim, bias=False)\n        self.norm = RMSNorm(cfg.gdn_value_dim)\n        self.A_log = nn.Parameter(torch.linspace(0, math.log(16), cfg.gdn_v_heads))\n        # Initial update time constants in [0.001, 0.1].\n        dt = torch.exp(torch.linspace(math.log(.001), math.log(.1), cfg.gdn_v_heads))\n        self.dt_bias = nn.Parameter(torch.log(torch.expm1(dt)))\n\n    def forward(self, x):\n        c = self.cfg\n        b, t, _ = x.shape\n        kw, vw = c.gdn_k_heads * c.gdn_key_dim, c.gdn_v_heads * c.gdn_value_dim\n        q, k, v = F.silu(self.conv(self.qkv(x))).split([kw, kw, vw], -1)\n        repeat = c.gdn_v_heads // c.gdn_k_heads\n        q = q.reshape(b, t, c.gdn_k_heads, c.gdn_key_dim).repeat_interleave(repeat, dim=2).transpose(1, 2)\n        k = k.reshape(b, t, c.gdn_k_heads, c.gdn_key_dim).repeat_interleave(repeat, dim=2).transpose(1, 2)\n        q, k = F.normalize(q.float(), dim=-1, eps=1e-6), F.normalize(k.float(), dim=-1, eps=1e-6)\n        v = v.reshape(b, t, c.gdn_v_heads, c.gdn_value_dim).transpose(1, 2)\n        log_alpha = -F.softplus(self.decay(x).float() + self.dt_bias.float()) * self.A_log.float().exp()\n        log_alpha = log_alpha.transpose(1, 2)\n        beta = self.beta(x).float().sigmoid().transpose(1, 2)\n        state = torch.zeros(b, c.gdn_v_heads, c.gdn_key_dim, c.gdn_value_dim, device=x.device)\n        outputs = []\n        for start in range(0, t, c.gdn_chunk):\n            stop = min(start + c.gdn_chunk, t)\n            y, state = delta_chunk(q[:, :, start:stop], k[:, :, start:stop], v[:, :, start:stop],\n                                  log_alpha[:, :, start:stop], beta[:, :, start:stop], state)\n            outputs.append(y)\n        y = torch.cat(outputs, dim=2).transpose(1, 2).to(x.dtype)\n        y = self.norm(y).reshape(b, t, vw) * self.gate(x).sigmoid()\n        return self.out(y)\n\n\ndef rope(x, theta):\n    t, d = x.shape[-2:]\n    positions = torch.arange(t, device=x.device).float()\n    inverse = theta ** (-torch.arange(0, d, 2, device=x.device).float() / d)\n    angle = positions[:, None] * inverse[None, :]\n    cos, sin = angle.cos().to(x.dtype), angle.sin().to(x.dtype)\n    left, right = x[..., :d // 2], x[..., d // 2:]\n    return torch.cat([left * cos - right * sin, right * cos + left * sin], -1)\n\n\nclass GQA(nn.Module):\n    def __init__(self, cfg, owner):\n        super().__init__()\n        self.cfg, self.owner = cfg, owner\n        self.q = nn.Linear(cfg.dim, cfg.q_heads * cfg.head_dim, bias=False)\n        self.gate = nn.Linear(cfg.dim, cfg.q_heads * cfg.head_dim, bias=False)\n        self.out = nn.Linear(cfg.q_heads * cfg.head_dim, cfg.dim, bias=False)\n        self.q_norm = RMSNorm(cfg.head_dim)\n        self.k_norm = RMSNorm(cfg.head_dim) if owner else None\n        self.k = nn.Linear(cfg.dim, cfg.kv_heads * cfg.head_dim, bias=False) if owner else None\n        self.v = nn.Linear(cfg.dim, cfg.kv_heads * cfg.head_dim, bias=False) if owner else None\n\n    def forward(self, x, kv=None):\n        c = self.cfg\n        b, t, _ = x.shape\n        q = rope(self.q_norm(self.q(x).reshape(b, t, c.q_heads, c.head_dim)).transpose(1, 2), c.rope_theta)\n        if self.owner:\n            k = rope(self.k_norm(self.k(x).reshape(b, t, c.kv_heads, c.head_dim)).transpose(1, 2), c.rope_theta)\n            v = self.v(x).reshape(b, t, c.kv_heads, c.head_dim).transpose(1, 2)\n            kv = (k, v)\n        if kv is None:\n            raise ValueError(\'KV reader requires source tensors.\')\n        # Explicit repeat is portable to Colab SDPA backends; logical KV stays shared.\n        k, v = [z.repeat_interleave(c.q_heads // c.kv_heads, dim=1) for z in kv]\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)\n        y = y.transpose(1, 2).reshape(b, t, c.q_heads * c.head_dim)\n        return self.out(y * self.gate(x).sigmoid()), kv\n\n\nclass Expert(nn.Module):\n    def __init__(self, dim, hidden):\n        super().__init__()\n        self.gate = nn.Linear(dim, hidden, bias=False)\n        self.up = nn.Linear(dim, hidden, bias=False)\n        self.down = nn.Linear(hidden, dim, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\n\nclass MoE(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.router = nn.Linear(cfg.dim, cfg.experts, bias=False)\n        self.experts = nn.ModuleList([Expert(cfg.dim, cfg.expert_dim) for _ in range(cfg.experts)])\n        self.shared = Expert(cfg.dim, cfg.expert_dim)\n\n    def forward(self, x):\n        shape = x.shape\n        flat = x.reshape(-1, self.cfg.dim)\n        with torch.autocast(device_type=x.device.type, enabled=False):\n            probabilities = F.linear(flat.float(), self.router.weight.float()).softmax(-1)\n        weights, selected = probabilities.topk(self.cfg.top_k, dim=-1)\n        weights = weights / weights.sum(-1, keepdim=True)\n        y = torch.zeros_like(flat)\n        for idx, expert in enumerate(self.experts):\n            token, slot = torch.where(selected == idx)\n            if token.numel():\n                out = expert(flat[token]) * weights[token, slot, None].to(x.dtype)\n                y = y.index_add(0, token, out.to(y.dtype))\n        counts = torch.bincount(selected.flatten(), minlength=self.cfg.experts).float()\n        fraction = counts / selected.numel()\n        auxiliary = self.cfg.experts * (fraction.detach() * probabilities.mean(0)).sum()\n        return (y + self.shared(flat)).reshape(shape), auxiliary, counts.detach()\n\n\nclass Block(nn.Module):\n    def __init__(self, cfg, index):\n        super().__init__()\n        self.full = (index + 1) % cfg.full_every == 0\n        self.owner = self.full and ((index + 1) // cfg.full_every - 1) % cfg.kv_group == 0\n        self.memory = Engram(cfg) if index == cfg.memory_layer else None\n        self.attn_gr, self.moe_gr = GatedResidual(cfg), GatedResidual(cfg)\n        self.mixer = GQA(cfg, self.owner) if self.full else GDN(cfg)\n        self.moe = MoE(cfg)\n\n    def forward(self, r, tokens, kv):\n        if self.memory is not None:\n            r = self.memory(tokens, r)\n        x, s = self.attn_gr.read(r)\n        if self.full:\n            y, kv = self.mixer(x, kv)\n        else:\n            y = self.mixer(x)\n        r = self.attn_gr.add(r, y, s)\n        x, s = self.moe_gr.read(r)\n        y, aux, counts = self.moe(x)\n        return self.moe_gr.add(r, y, s), kv, aux, counts\n\n\nclass NextMoE(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.config = cfg\n        self.embedding = nn.Embedding(cfg.vocab_size, cfg.dim)\n        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.layers)])\n        self.final_gr = GatedResidual(cfg, write=False)\n        self.apply(self._init)\n        # Zero initial write projections => scalar write 1. Small mixer outputs\n        # keep a newly initialized multi-stream residual stable.\n        for block in self.blocks:\n            nn.init.zeros_(block.attn_gr.write.weight)\n            nn.init.zeros_(block.moe_gr.write.weight)\n            nn.init.normal_(block.mixer.out.weight, std=.02 / math.sqrt(2 * cfg.layers))\n        self.last_counts = None\n\n    @staticmethod\n    def _init(module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=.02)\n\n    def features(self, tokens):\n        r = self.embedding(tokens)\n        if torch.is_autocast_enabled(tokens.device.type):\n            r = r.to(torch.get_autocast_dtype(tokens.device.type))\n        r = r.unsqueeze(-2).expand(-1, -1, self.config.branches, -1).clone()\n        kv, auxiliary, counts = None, [], []\n        for block in self.blocks:\n            if self.training and self.config.checkpoint_blocks and torch.is_grad_enabled():\n                r, kv, aux, count = checkpoint(block, r, tokens, kv, use_reentrant=False)\n            else:\n                r, kv, aux, count = block(r, tokens, kv)\n            auxiliary.append(aux)\n            counts.append(count)\n        self.last_counts = torch.stack(counts)\n        return self.final_gr.read(r)[0], torch.stack(auxiliary).mean()\n\n    def forward(self, tokens, targets=None, loss_chunk=64):\n        hidden, auxiliary = self.features(tokens)\n        if targets is None:\n            return F.linear(hidden, self.embedding.weight)\n        # Chunk and checkpoint the softmax/head to avoid B*T*V live logits.\n        total = hidden.new_zeros((), dtype=torch.float32)\n        for start in range(0, tokens.shape[1], loss_chunk):\n            h = hidden[:, start:start + loss_chunk]\n            y = targets[:, start:start + loss_chunk]\n            def loss_fn(hh, yy):\n                logits = F.linear(hh, self.embedding.weight)\n                return F.cross_entropy(logits.float().reshape(-1, self.config.vocab_size),\n                                       yy.reshape(-1), reduction=\'sum\')\n            loss = checkpoint(loss_fn, h, y, use_reentrant=False) if self.training else loss_fn(h, y)\n            total = total + loss\n        return {\'ce\': total / targets.numel(), \'aux\': auxiliary}\n\n    @torch.no_grad()\n    def generate(self, tokens, max_new_tokens=64, temperature=.8, top_k=40, context=512, valid_vocab=None):\n        was_training = self.training\n        self.eval()\n        try:\n            for _ in range(max_new_tokens):\n                hidden, _ = self.features(tokens[:, -context:])\n                logits = F.linear(hidden[:, -1], self.embedding.weight).float()\n                if valid_vocab is not None:\n                    logits[:, valid_vocab:] = float(\'-inf\')\n                logits[:, self.config.pad_id] = float(\'-inf\')\n                if temperature <= 0:\n                    nxt = logits.argmax(-1, keepdim=True)\n                else:\n                    logits = logits / temperature\n                    threshold = logits.topk(min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]\n                    logits = logits.masked_fill(logits < threshold, float(\'-inf\'))\n                    nxt = torch.multinomial(logits.softmax(-1), 1)\n                tokens = torch.cat([tokens, nxt], -1)\n                if tokens.shape[0] == 1 and nxt.item() == self.config.eos_id:\n                    break\n            return tokens\n        finally:\n            self.train(was_training)\n', 'next_moe/precision.py': '"""FP16/BF16 model parameters with FP32 master weights for stable Adam updates."""\nimport torch\n\n\nclass MasterWeights:\n    def __init__(self, parameters):\n        self.model = list(parameters)\n        self.master = [torch.nn.Parameter(p.detach().float().clone(), requires_grad=p.requires_grad)\n                       for p in self.model]\n\n    def copy_gradients(self):\n        # GradScaler operates on FP32 masters, avoiding unscale_ on FP16 gradients.\n        for model, master in zip(self.model, self.master):\n            master.grad = None if model.grad is None else model.grad.detach().float()\n            model.grad = None  # release low-precision gradients as we convert\n\n    @torch.no_grad()\n    def copy_to_model(self):\n        for model, master in zip(self.model, self.master):\n            model.copy_(master)\n\n    def zero_grad(self):\n        for model in self.model:\n            model.grad = None\n', 'next_moe/train.py': '"""Single-GPU test trainer with autocast, checkpointing, evaluation and resume."""\nfrom contextlib import nullcontext\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nimport torch\nfrom .data import TokenBatches\nfrom .model import ModelConfig, NextMoE\nfrom .precision import MasterWeights\n\n\n@dataclass\nclass TrainConfig:\n    steps: int = 300  # total optimizer updates, including resumed updates\n    sequence_length: int = 128\n    microbatch: int = 1\n    accumulation: int = 4\n    learning_rate: float = 2e-4\n    warmup: int = 20\n    auxiliary_weight: float = .01\n    weight_decay: float = .1\n    grad_clip: float = 1.0\n    eval_every: int = 50\n    eval_batches: int = 4\n    save_every: int = 100\n    log_every: int = 10\n    optimizer: str = \'adamw\'  # optional \'adamw8bit\'; no CPU offload\n    seed: int = 1234\n    precision: str = \'fp32\'  # fp32 / fp16_master / bf16_master\n    restart_warmup: int = 100\n\n\nclass Trainer:\n    def __init__(self, model_config, train_config, data_dir, output_dir, device=None, resume=None,\n                 resume_mode=\'exact\', resume_tokenizer=None, checkpoint_dir=None,\n                 persistent_weights=False, disk_budget_gib=None):\n        self.cfg, self.model_config = train_config, model_config\n        self.device = torch.device(device or (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n        if min(train_config.steps, train_config.accumulation, train_config.microbatch,\n               train_config.sequence_length, train_config.eval_batches, train_config.log_every,\n               train_config.eval_every, train_config.save_every) < 1:\n            raise ValueError(\'Step/batch/interval values must be positive.\')\n        if train_config.optimizer not in (\'adamw\', \'adamw8bit\'):\n            raise ValueError(\'Unknown optimizer.\')\n        self.output = Path(output_dir)\n        self.output.mkdir(parents=True, exist_ok=True)\n        self.checkpoint_dir = Path(checkpoint_dir or output_dir)\n        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)\n        self.persistent_weights = persistent_weights\n        self.disk_budget_gib = disk_budget_gib\n        self.log_path = self.checkpoint_dir / \'metrics.jsonl\'\n        import shutil\n        if self.log_path != self.output / \'metrics.jsonl\' and not self.log_path.exists() and (self.output / \'metrics.jsonl\').exists():\n            shutil.copy2(self.output / \'metrics.jsonl\', self.log_path)\n        self.data_dir = Path(data_dir)\n        meta = json.loads((self.data_dir / \'metadata.json\').read_text())\n        if meta[\'actual_vocab\'] > model_config.vocab_size:\n            raise ValueError(\'Tokenizer vocabulary exceeds model embedding size.\')\n        if meta[\'special_ids\'][\'<eos>\'] != model_config.eos_id or meta[\'special_ids\'][\'<pad>\'] != model_config.pad_id:\n            raise ValueError(\'Tokenizer special IDs do not match model configuration.\')\n        self.data_signature = self._data_signature()\n        self.tokenizer_signature = hashlib.sha256((self.data_dir / \'tokenizer.json\').read_bytes()).hexdigest()\n        torch.manual_seed(train_config.seed)\n        if self.device.type == \'cuda\':\n            torch.cuda.manual_seed_all(train_config.seed)\n        self.batches = TokenBatches(data_dir, train_config.sequence_length,\n                                   train_config.microbatch, train_config.seed)\n        if train_config.precision not in (\'fp32\', \'fp16_master\', \'bf16_master\'):\n            raise ValueError(\'Unknown parameter precision.\')\n        self.dtype = (torch.bfloat16 if self.device.type == \'cuda\' and torch.cuda.is_bf16_supported(including_emulation=False)\n                      else torch.float16)\n        parameter_dtype = {\'fp32\': torch.float32, \'fp16_master\': torch.float16,\n                           \'bf16_master\': torch.bfloat16}[train_config.precision]\n        if parameter_dtype == torch.bfloat16 and self.device.type == \'cuda\' and not torch.cuda.is_bf16_supported(including_emulation=False):\n            raise ValueError(\'This CUDA GPU does not support BF16. Use fp16_master or fp32.\')\n        if parameter_dtype != torch.float32:\n            self.dtype = parameter_dtype\n        self.model = NextMoE(model_config).to(self.device)\n        self.masters = MasterWeights(self.model.parameters()) if parameter_dtype != torch.float32 else None\n        if self.masters:\n            self.model.to(dtype=parameter_dtype)\n            self.masters.model = list(self.model.parameters())\n        self.optim_parameters = self.masters.master if self.masters else list(self.model.parameters())\n        if train_config.optimizer == \'adamw8bit\':\n            if self.device.type != \'cuda\':\n                raise ValueError(\'8-bit optimizer needs a supported CUDA GPU.\')\n            import bitsandbytes as bnb\n            self.optimizer = bnb.optim.AdamW8bit(self.optim_parameters,\n                                                lr=train_config.learning_rate,\n                                                weight_decay=train_config.weight_decay)\n        else:\n            self.optimizer = torch.optim.AdamW(self.optim_parameters, lr=train_config.learning_rate,\n                                               weight_decay=train_config.weight_decay, foreach=False)\n        self.scaler = torch.amp.GradScaler(self.device.type, init_scale=4096,\n                                          enabled=self.dtype == torch.float16 and\n                                          (self.device.type == \'cuda\' or self.masters is not None))\n        self.restart_step = None\n        self.last_saved_step = -1\n        self.routing_total = torch.zeros(model_config.layers, model_config.experts, device=self.device)\n        self.step, self.tokens_seen = 0, 0\n        self.last_counts = None\n        if resume:\n            self.load(resume, mode=resume_mode, tokenizer_path=resume_tokenizer)\n        (self.output / \'model_config.json\').write_text(json.dumps(model_config.to_dict(), indent=2))\n        (self.output / \'train_config.json\').write_text(json.dumps(asdict(train_config), indent=2))\n        import shutil\n        shutil.copy2(self.data_dir / \'tokenizer.json\', self.output / \'tokenizer.json\')\n        shutil.copy2(self.data_dir / \'metadata.json\', self.output / \'data_metadata.json\')\n        print(f\'Parameters: {sum(p.numel() for p in self.model.parameters()):,}; device={self.device}; \'\n              f\'parameters={parameter_dtype}; autocast={self.dtype if self.device.type == "cuda" else "off"}; step={self.step}\')\n\n    def _data_signature(self):\n        digest = hashlib.sha256()\n        for name in [\'tokenizer.json\', \'metadata.json\', \'train_uk.bin\', \'train_en.bin\', \'val_uk.bin\', \'val_en.bin\']:\n            with (self.data_dir / name).open(\'rb\') as handle:\n                while chunk := handle.read(1 << 20):\n                    digest.update(chunk)\n        return digest.hexdigest()\n\n    def autocast(self):\n        return (torch.autocast(\'cuda\', dtype=self.dtype) if self.device.type == \'cuda\' else nullcontext())\n\n    def learning_rate(self):\n        c = self.cfg\n        if self.step < c.warmup:\n            lr = c.learning_rate * (self.step + 1) / max(c.warmup, 1)\n        else:\n            progress = (self.step - c.warmup) / max(c.steps - c.warmup, 1)\n            lr = c.learning_rate * (.1 + .9 * .5 * (1 + math.cos(math.pi * min(progress, 1))))\n        if self.restart_step is not None and c.restart_warmup > 0:\n            lr *= min(1.0, (self.step - self.restart_step + 1) / c.restart_warmup)\n        return lr\n\n    @torch.no_grad()\n    def evaluate(self):\n        self.model.eval()\n        result = {}\n        for index, language in enumerate((\'uk\', \'en\')):\n            rng = torch.Generator().manual_seed(self.cfg.seed + 999 + index)\n            losses = []\n            for _ in range(self.cfg.eval_batches):\n                x, y = self.batches.get(\'val\', self.device, language, rng)\n                with self.autocast():\n                    losses.append(float(self.model(x, y)[\'ce\']))\n            result[f\'val_ce_{language}\'] = sum(losses) / len(losses)\n            result[f\'val_ppl_{language}\'] = math.exp(min(result[f\'val_ce_{language}\'], 20))\n        self.model.train()\n        return result\n\n    @staticmethod\n    def payload_bytes(payload):\n        seen = set()\n        def size(value):\n            if isinstance(value, torch.Tensor):\n                storage = value.untyped_storage()\n                key = (str(value.device), storage.data_ptr())\n                if key in seen:\n                    return 0\n                seen.add(key)\n                return storage.nbytes()\n            if isinstance(value, dict):\n                return sum(size(v) for v in value.values())\n            if isinstance(value, (list, tuple)):\n                return sum(size(v) for v in value)\n            return 0\n        return size(payload)\n\n    def _write(self, payload, destination, budget=None):\n        temporary = destination.with_suffix(\'.pt.tmp\')\n        estimated = int(self.payload_bytes(payload) * 1.05) + (1 << 20)\n        if budget is not None:\n            used = sum(p.stat().st_size for p in self.output.rglob(\'*\') if p.is_file())\n            if used + estimated > budget * 2**30:\n                raise OSError(f\'Run disk budget exceeded: existing {used/2**30:.2f} GiB + temporary \'\n                              f\'{estimated/2**30:.2f} GiB > {budget:.2f} GiB. Previous snapshot is preserved.\')\n        try:\n            torch.save(payload, temporary)\n            os.replace(temporary, destination)\n        finally:\n            if temporary.exists():\n                temporary.unlink()\n        print(f\'Saved {destination} ({destination.stat().st_size / 2**30:.2f} GiB)\')\n        return destination\n\n    def snapshot(self):\n        return {\'kind\': \'weights\', \'model\': self.model.state_dict(),\n                \'step\': self.step, \'tokens_seen\': self.tokens_seen,\n                \'model_config\': self.model_config.to_dict(), \'train_config\': asdict(self.cfg),\n                \'data_signature\': self.data_signature, \'tokenizer_signature\': self.tokenizer_signature}\n\n    def save(self, filename=\'last.pt\'):\n        checkpoint = {**self.snapshot(), \'kind\': \'full\',\n                      \'optimizer\': self.optimizer.state_dict(), \'scaler\': self.scaler.state_dict(),\n                      \'restart_step\': self.restart_step, \'routing_total\': self.routing_total,\n                      \'master_weights\': [p.detach() for p in self.masters.master] if self.masters else None,\n                      \'batch_rng\': self.batches.rng.get_state(), \'torch_rng\': torch.get_rng_state(),\n                      \'cuda_rng\': torch.cuda.get_rng_state_all() if self.device.type == \'cuda\' else []}\n        destination = self._write(checkpoint, self.checkpoint_dir / filename,\n                                  self.disk_budget_gib if self.checkpoint_dir == self.output else None)\n        if self.persistent_weights:\n            self._write(self.snapshot(), self.output / \'weights.pt\', self.disk_budget_gib)\n        if self.log_path != self.output / \'metrics.jsonl\' and self.log_path.exists():\n            import shutil\n            shutil.copy2(self.log_path, self.output / \'metrics.jsonl\')\n        self.last_saved_step = self.step\n        return destination\n\n    def load(self, path, mode=\'exact\', tokenizer_path=None):\n        if mode not in (\'exact\', \'warm\'):\n            raise ValueError(\'Resume mode must be exact or warm.\')\n        saved = torch.load(path, map_location=\'cpu\', weights_only=True, mmap=True)\n        if saved[\'model_config\'] != self.model_config.to_dict():\n            raise ValueError(\'Resume requires identical model configuration.\')\n        if mode == \'exact\':\n            if \'optimizer\' not in saved:\n                raise ValueError(\'Weights-only snapshot requires explicit warm restart.\')\n            if saved[\'data_signature\'] != self.data_signature:\n                raise ValueError(\'Exact resume requires exactly the same tokenizer and corpus.\')\n            old = saved[\'train_config\']\n            for name in (\'optimizer\', \'sequence_length\', \'microbatch\', \'accumulation\', \'learning_rate\',\n                         \'warmup\', \'weight_decay\', \'grad_clip\', \'auxiliary_weight\', \'seed\', \'precision\', \'restart_warmup\'):\n                if old.get(name, getattr(TrainConfig(), name)) != getattr(self.cfg, name):\n                    raise ValueError(f\'Exact resume changed {name}. Use a new phase with explicit warm restart.\')\n        else:\n            signature = saved.get(\'tokenizer_signature\')\n            if signature is None and tokenizer_path is not None:\n                signature = hashlib.sha256(Path(tokenizer_path).read_bytes()).hexdigest()\n            if signature is None and saved.get(\'data_signature\') == self.data_signature:\n                signature = self.tokenizer_signature  # legacy checkpoint on identical corpus\n            if signature != self.tokenizer_signature:\n                raise ValueError(\'Warm restart requires the SAME tokenizer; do not retrain BPE.\')\n        self.model.load_state_dict(saved[\'model\'])\n        if self.masters:\n            with torch.no_grad():\n                values = saved.get(\'master_weights\') if mode == \'exact\' else None\n                if values is None:\n                    # Use full saved FP32 values when migrating a legacy checkpoint.\n                    values = [saved[\'model\'][name] for name, _ in self.model.named_parameters()]\n                for master, value in zip(self.masters.master, values):\n                    master.copy_(value)\n            self.masters.copy_to_model()\n        self.step, self.tokens_seen = saved[\'step\'], saved[\'tokens_seen\']\n        if mode == \'exact\':\n            self.optimizer.load_state_dict(saved[\'optimizer\'])\n            self.scaler.load_state_dict(saved[\'scaler\'])\n            self.batches.rng.set_state(saved[\'batch_rng\'])\n            torch.set_rng_state(saved[\'torch_rng\'])\n            if self.device.type == \'cuda\' and saved[\'cuda_rng\']:\n                torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n            self.restart_step = saved.get(\'restart_step\')\n            if \'routing_total\' in saved:\n                self.routing_total.copy_(saved[\'routing_total\'])\n            self.last_saved_step = self.step\n            print(f\'Exact resume at step {self.step}; a changed total steps changes future cosine schedule.\')\n        else:\n            self.restart_step = self.step\n            print(f\'WARM RESTART at step {self.step}: optimizer/RNG reset, \'\n                  f\'{self.cfg.restart_warmup}-update LR ramp; not an exact continuation.\')\n\n    def zero_grad(self):\n        self.optimizer.zero_grad(set_to_none=True)\n        if self.masters:\n            self.masters.zero_grad()\n\n    def run(self, max_updates=None):\n        c = self.cfg\n        self.model.train()\n        self.zero_grad()\n        if self.device.type == \'cuda\':\n            torch.cuda.reset_peak_memory_stats()\n        last_log, window_tokens = time.perf_counter(), 0\n        overflow_retries = 0\n        stop_step = min(c.steps, self.step + max_updates) if max_updates is not None else c.steps\n        try:\n            while self.step < stop_step:\n                update_started = time.perf_counter()\n                data_seconds = 0.0\n                lr = self.learning_rate()\n                for group in self.optimizer.param_groups:\n                    group[\'lr\'] = lr\n                ce_total, aux_total, counts = 0.0, 0.0, None\n                batch_rng_start = self.batches.rng.get_state()\n                for _ in range(c.accumulation):\n                    data_started = time.perf_counter()\n                    x, y = self.batches.get(\'train\', self.device)\n                    data_seconds += time.perf_counter() - data_started\n                    with self.autocast():\n                        out = self.model(x, y)\n                        loss = (out[\'ce\'] + c.auxiliary_weight * out[\'aux\']) / c.accumulation\n                    if not bool(torch.isfinite(loss)):\n                        raise FloatingPointError(\'Non-finite loss. Check data, LR and precision.\')\n                    self.scaler.scale(loss).backward()\n                    ce_total += float(out[\'ce\'].detach()) / c.accumulation\n                    aux_total += float(out[\'aux\'].detach()) / c.accumulation\n                    now_counts = self.model.last_counts\n                    counts = now_counts.clone() if counts is None else counts + now_counts\n                if self.masters:\n                    self.masters.copy_gradients()\n                self.scaler.unscale_(self.optimizer)\n                norm = torch.nn.utils.clip_grad_norm_(self.optim_parameters, c.grad_clip)\n                if not bool(torch.isfinite(norm)):\n                    if self.scaler.is_enabled() and overflow_retries < 8:\n                        # unscale_ registered infs: scaler.step skips the optimizer update.\n                        # Restore sampler so the smaller scale retries the same microbatches.\n                        old_scale = self.scaler.get_scale()\n                        self.scaler.step(self.optimizer)\n                        self.scaler.update()\n                        self.zero_grad()\n                        self.batches.rng.set_state(batch_rng_start)\n                        overflow_retries += 1\n                        print(f\'FP16 overflow: retry {overflow_retries}, scale {old_scale} -> {self.scaler.get_scale()}\')\n                        continue\n                    raise FloatingPointError(\'Persistent non-finite gradients. Reduce LR or use BF16 if available.\')\n                overflow_retries = 0\n                self.scaler.step(self.optimizer)\n                self.scaler.update()\n                if self.masters:\n                    self.masters.copy_to_model()\n                self.zero_grad()\n                self.step += 1\n                seen = c.microbatch * c.sequence_length * c.accumulation\n                self.tokens_seen += seen\n                window_tokens += seen\n                self.last_counts = counts\n                self.routing_total += counts\n                if self.device.type == \'cuda\':\n                    torch.cuda.synchronize()\n                update_seconds = time.perf_counter() - update_started\n                record = {\'step\': self.step, \'train_ce\': ce_total, \'router_aux\': aux_total,\n                          \'lr\': lr, \'tokens_seen\': self.tokens_seen, \'grad_norm\': float(norm),\n                          \'tokens_per_second_training_step\': seen / max(update_seconds, 1e-6),\n                          \'data_loading_seconds\': data_seconds,\n                          \'unselected_expert_fraction_phase\': float((self.routing_total == 0).float().mean()),\n                          \'max_expert_to_mean_load_phase\': float((self.routing_total.max(-1).values / self.routing_total.mean(-1).clamp_min(1)).max()),\n                          \'unselected_expert_fraction_this_step\': float((counts == 0).float().mean()),\n                          \'max_expert_to_mean_load\': float((counts.max(-1).values / counts.mean(-1).clamp_min(1)).max())}\n                if self.step % c.log_every == 0 or self.step == 1:\n                    if self.device.type == \'cuda\':\n                        torch.cuda.synchronize()\n                        record[\'peak_allocated_GiB\'] = torch.cuda.max_memory_allocated() / 2**30\n                        record[\'peak_reserved_GiB\'] = torch.cuda.max_memory_reserved() / 2**30\n                    record[\'tokens_per_second_including_interval_overhead\'] = window_tokens / max(time.perf_counter() - last_log, 1e-6)\n                    print(json.dumps(record))\n                    last_log, window_tokens = time.perf_counter(), 0\n                if self.step % c.eval_every == 0:\n                    record.update(self.evaluate())\n                    print(json.dumps({\'step\': self.step, **{k: v for k, v in record.items() if k.startswith(\'val_\')}}))\n                with self.log_path.open(\'a\') as handle:\n                    handle.write(json.dumps(record) + \'\\n\')\n                if self.step % c.save_every == 0:\n                    self.save()\n        except KeyboardInterrupt:\n            # If interrupt occurred inside accumulation, partial gradients are not saved.\n            self.zero_grad()\n            print(\'Interrupted: saving the last completed optimizer step; partial microbatch RNG may have advanced.\')\n        except torch.cuda.OutOfMemoryError:\n            print(\'CUDA OOM: reduce sequence_length/microbatch and start a NEW run; leave checkpointing enabled.\')\n            raise\n        if self.last_saved_step != self.step:\n            self.save()\n        return self.evaluate()\n\n    def export_weights(self):\n        path = self.output / \'model.pt\'\n        return self._write(self.snapshot(), path, self.disk_budget_gib)\n'}
SOURCE_HASHES = {'next_moe/__init__.py': '783e1386ddb7244ccd199a59860580bd34bc58d92c79587d7d1c39e48f1dc975', 'next_moe/corpus.py': '1d95400ae6f85aa525aeb38e9d3f5ad0ae94aeaa110b951c35b1f575b5c847ab', 'next_moe/data.py': 'dfa0eb2c2fd00243a4c3cc8b15d4531096bdcb1b3b266416dc6281cd0a9efd6b', 'next_moe/model.py': '64356c659de22fea892a6effaafd8c51ee9e2820ae2df1fe18a9c9f1cee98a24', 'next_moe/precision.py': 'a9e5ca02ac02bf5285d325aec40572b2c729f656d3deb698bb5e5f33dcf59ec1', 'next_moe/train.py': '712d3c96d7b0d37cb8a12b973c84c581ac765bc60bccf63e77be32097d2888e5'}
for name, content in SOURCES.items():
    target = CODE_DIR/name; target.parent.mkdir(parents=True, exist_ok=True); target.write_text(content)
if str(CODE_DIR) not in sys.path: sys.path.insert(0, str(CODE_DIR))
from next_moe.model import ModelConfig, NextMoE
from next_moe.train import TrainConfig, Trainer
from next_moe.corpus import prepare_continuation_data, install_domain_evaluation, install_phase_schedule
from tokenizers import Tokenizer
exec('"""Live Colab efficiency tools. No model reload, external services or multi-GPU.\n\nBenchmark executes short optimizer updates then restores model/master/optimizer,\nprogress and RNG after every candidate. CPU backup uses ~full-checkpoint RAM.\n"""\nfrom dataclasses import asdict\nimport copy\nimport json\nimport math\nfrom pathlib import Path\nimport statistics\nimport tempfile\nimport torch\n\n\ndef _cpu_copy(value):\n    if isinstance(value, torch.Tensor):\n        return value.detach().to(device=\'cpu\', copy=True)\n    if isinstance(value, dict):\n        return {key: _cpu_copy(item) for key, item in value.items()}\n    if isinstance(value, list):\n        return [_cpu_copy(item) for item in value]\n    if isinstance(value, tuple):\n        return tuple(_cpu_copy(item) for item in value)\n    return copy.deepcopy(value)\n\n\ndef fix_evaluation_microbatch(trainer, batch=1):\n    """Keep fixed validation windows when changing the TRAIN microbatch."""\n    if getattr(trainer, \'_fixed_evaluation_installed\', False):\n        trainer._evaluation_batch = batch\n        return\n    original = trainer.evaluate\n    trainer._evaluation_batch = batch\n    def fixed_evaluate():\n        previous = trainer.batches.batch\n        trainer.batches.batch = trainer._evaluation_batch\n        try:\n            return original()\n        finally:\n            trainer.batches.batch = previous\n    trainer.evaluate = fixed_evaluate\n    trainer._fixed_evaluation_installed = True\n\n\ndef configure_live_batch(trainer, microbatch, accumulation, checkpoint_blocks=None):\n    """Repartition the SAME tokens/update, retaining optimizer and LR schedule."""\n    if min(microbatch, accumulation) < 1:\n        raise ValueError(\'Batch sizes must be positive integers.\')\n    if microbatch * accumulation != trainer.cfg.microbatch * trainer.cfg.accumulation:\n        raise ValueError(\'This helper preserves effective batch; microbatch × accumulation must stay equal.\')\n    fix_evaluation_microbatch(trainer, 1)\n    trainer.cfg.microbatch, trainer.cfg.accumulation = microbatch, accumulation\n    trainer.batches.batch = microbatch\n    if checkpoint_blocks is not None:\n        trainer.model.config.checkpoint_blocks = checkpoint_blocks\n    (trainer.output/\'train_config.json\').write_text(json.dumps(asdict(trainer.cfg), indent=2))\n    (trainer.output/\'model_config.json\').write_text(json.dumps(trainer.model.config.to_dict(), indent=2))\n\n\ndef benchmark_batch_layout(trainer, units_per_hour=8.9, warmup_updates=2,\n                           measured_updates=10, apply_best=True):\n    """Non-destructive timing comparison with equal effective batch.\n\n    Uses host RAM for a backup; short candidate updates are rolled back. No Drive\n    writes or validation during timing. Full Adam states must already be allocated.\n    Absolute CUDA speed must be measured on the user\'s actual GPU.\n    """\n    if (units_per_hour is not None and units_per_hour <= 0) or measured_updates < 1 or warmup_updates < 0:\n        raise ValueError(\'Invalid benchmark settings.\')\n    updates = warmup_updates + measured_updates\n    if trainer.cfg.steps - trainer.step < updates:\n        raise ValueError(\'Too close to the current training goal for this benchmark. Finish the stage first.\')\n    trainer.zero_grad()\n    if not trainer.optimizer.state:\n        raise ValueError(\'Run at least one real update first so Adam states are allocated.\')\n    fix_evaluation_microbatch(trainer, 1)\n    base_cfg = copy.deepcopy(trainer.cfg)\n    base_checkpoint = trainer.model.config.checkpoint_blocks\n    saved = {\'model\': _cpu_copy(trainer.model.state_dict()),\n             \'master\': _cpu_copy([p for p in trainer.masters.master]) if trainer.masters else None,\n             \'optimizer\': _cpu_copy(trainer.optimizer.state_dict()),\n             \'scaler\': copy.deepcopy(trainer.scaler.state_dict()),\n             \'step\': trainer.step, \'tokens\': trainer.tokens_seen,\n             \'batch_rng\': trainer.batches.rng.get_state().clone(),\n             \'torch_rng\': torch.get_rng_state().clone(),\n             \'cuda_rng\': torch.cuda.get_rng_state_all() if trainer.device.type == \'cuda\' else [],\n             \'routing\': _cpu_copy(trainer.routing_total),\n             \'last_counts\': _cpu_copy(trainer.last_counts),\n             \'last_saved_step\': trainer.last_saved_step,\n             \'restart_step\': trainer.restart_step, \'training\': trainer.model.training}\n    original_save, original_evaluate, original_log = trainer.save, trainer.evaluate, trainer.log_path\n    effective = base_cfg.microbatch * base_cfg.accumulation\n    microbatches = sorted(set([base_cfg.microbatch] + [v for v in (1, 2, 4) if effective % v == 0]))\n    candidates = [(m, effective // m, base_checkpoint) for m in microbatches]\n    if base_checkpoint:\n        candidates.append((max(microbatches), effective // max(microbatches), False))\n    results = []\n\n    def restore():\n        trainer.zero_grad()\n        trainer.cfg = copy.deepcopy(base_cfg)\n        trainer.batches.batch = base_cfg.microbatch\n        trainer.model.config.checkpoint_blocks = base_checkpoint\n        trainer.model.load_state_dict(saved[\'model\'])\n        if trainer.masters:\n            with torch.no_grad():\n                for parameter, value in zip(trainer.masters.master, saved[\'master\']):\n                    parameter.copy_(value)\n        # load_state_dict may reuse CPU tensors (including Adam\'s step scalar).\n        # Give every candidate its own copy so updates cannot mutate the backup.\n        trainer.optimizer.load_state_dict(copy.deepcopy(saved[\'optimizer\']))\n        trainer.scaler.load_state_dict(saved[\'scaler\'])\n        trainer.step, trainer.tokens_seen = saved[\'step\'], saved[\'tokens\']\n        trainer.last_saved_step, trainer.restart_step = saved[\'last_saved_step\'], saved[\'restart_step\']\n        trainer.batches.rng.set_state(saved[\'batch_rng\'])\n        torch.set_rng_state(saved[\'torch_rng\'])\n        if saved[\'cuda_rng\']:\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        trainer.routing_total.copy_(saved[\'routing\'])\n        trainer.last_counts = saved[\'last_counts\'].to(trainer.device) if saved[\'last_counts\'] is not None else None\n        trainer.model.train(saved[\'training\'])\n        if trainer.device.type == \'cuda\':\n            torch.cuda.synchronize()\n            torch.cuda.empty_cache()\n\n    try:\n        with tempfile.TemporaryDirectory(prefix=\'next-moe-benchmark-\') as directory:\n            trainer.save = lambda *args, **kwargs: None\n            trainer.evaluate = lambda: {}\n            for index, (microbatch, accumulation, checkpointing) in enumerate(candidates):\n                restore()\n                trainer.cfg.microbatch, trainer.cfg.accumulation = microbatch, accumulation\n                trainer.batches.batch = microbatch\n                trainer.model.config.checkpoint_blocks = checkpointing\n                trainer.log_path = Path(directory)/f\'{index}.jsonl\'\n                print(f\'BENCHMARK: microbatch={microbatch}, accumulation={accumulation}, \'\n                      f\'block_checkpointing={checkpointing}; updates will be rolled back.\')\n                trainer.run(max_updates=updates)\n                records = [json.loads(line) for line in trainer.log_path.read_text().splitlines()]\n                rates = [r[\'tokens_per_second_training_step\'] for r in records[warmup_updates:]]\n                speed = statistics.median(rates)\n                result = {\'microbatch\': microbatch, \'accumulation\': accumulation,\n                          \'checkpoint_blocks\': checkpointing,\n                          \'effective_tokens_per_update\': microbatch*accumulation*trainer.cfg.sequence_length,\n                          \'median_training_tokens_per_second\': speed,\n                          \'units_per_hour_assumed\': units_per_hour,\n                          \'training_tokens_per_unit\': speed*3600/units_per_hour if units_per_hour else None}\n                if trainer.device.type == \'cuda\':\n                    result[\'peak_allocated_GiB\'] = torch.cuda.max_memory_allocated()/2**30\n                    result[\'peak_reserved_GiB\'] = torch.cuda.max_memory_reserved()/2**30\n                results.append(result)\n                print(json.dumps(result))\n    finally:\n        restore()\n        trainer.save, trainer.evaluate, trainer.log_path = original_save, original_evaluate, original_log\n    if apply_best and results:\n        winner = max(results, key=lambda r: r[\'median_training_tokens_per_second\'])\n        configure_live_batch(trainer, winner[\'microbatch\'], winner[\'accumulation\'], winner[\'checkpoint_blocks\'])\n        print(\'Applied fastest measured layout; model/optimizer/progress/RNG restored. \'\n              \'Update MICROBATCH, GRAD_ACCUM and CHECKPOINT_BLOCKS in notebook settings before a future reload.\')\n    report = {\'device\': str(trainer.device), \'rolled_back_to_step\': trainer.step,\n              \'tokens_seen\': trainer.tokens_seen, \'results\': results,\n              \'scope\': \'Training-step timing; exclude eval/save overhead. Not a GPU utilization or quality benchmark.\'}\n    (trainer.output/\'efficiency-benchmark.json\').write_text(json.dumps(report, indent=2))\n    return results\n\n\ndef extend_token_budget(trainer, target_tokens, ramp_updates=100):\n    """Extend goal in the live session, preserving optimizer; ramp avoids LR jump."""\n    if target_tokens <= trainer.tokens_seen or ramp_updates < 1:\n        raise ValueError(\'Target must exceed processed tokens; use a positive ramp.\')\n    batch_tokens = trainer.cfg.sequence_length * trainer.cfg.microbatch * trainer.cfg.accumulation\n    trainer.cfg.steps = trainer.step + math.ceil((target_tokens-trainer.tokens_seen)/batch_tokens)\n    trainer.restart_step = trainer.step  # LR ramp only; no optimizer reset\n    trainer.cfg.restart_warmup = ramp_updates\n    (trainer.output/\'train_config.json\').write_text(json.dumps(asdict(trainer.cfg), indent=2))\n    print(f\'New goal ~{target_tokens:,} total tokens / step {trainer.cfg.steps}; \'\n          f\'optimizer retained, {ramp_updates}-update LR ramp.\')\n')


## Перевірка базової моделі та незмінного tokenizer

При першому запуску +20M відраховуються від фактичного tokens_seen checkpoint,
а не від припущення, що він рівно 30,000,000. Phase target зберігається на Drive.
Повторне відкриття не додає ще 20M і не повторює extend_token_budget.


In [4]:
base_checkpoint = BASE_RUN/'weights.pt'
for path in (base_checkpoint, OLD_DATA/'tokenizer.json', OLD_DATA/'metadata.json',
             OLD_DATA/'val_uk.bin', OLD_DATA/'val_en.bin'):
    if not path.exists():
        raise FileNotFoundError(f'{path}. Set BASE_RUN_NAME/BASE_DATA_NAME to your saved 30M run.')
phase_path = RUN_DIR/'phase.json'
if phase_path.exists():
    phase = json.loads(phase_path.read_text())
    if phase['additional_tokens'] != ADDITIONAL_TOKENS:
        raise ValueError('Existing phase has a different budget. Use a new RUN_NAME.')
    base_sha = phase['base_weights_sha256']
    if str(BASE_RUN) != phase['base_run'] or hashlib.sha256(base_checkpoint.read_bytes()).hexdigest() != base_sha:
        raise ValueError('Frozen base checkpoint changed. Restore it or use a NEW RUN_NAME.')
else:
    base_sha = hashlib.sha256(base_checkpoint.read_bytes()).hexdigest()
    base_local = LOCAL_DIR/'base-weights.pt'
    shutil.copy2(base_checkpoint, base_local)
    header = torch.load(base_local, map_location='cpu', weights_only=True, mmap=True)
    if not 27_000_000 <= header['tokens_seen'] <= 32_000_000:
        raise ValueError('Expected checkpoint near 30M; verify BASE_RUN before continuing.')
    phase = {'initial_step':header['step'], 'initial_tokens':header['tokens_seen'],
             'additional_tokens':ADDITIONAL_TOKENS,
             'target_tokens':header['tokens_seen']+ADDITIONAL_TOKENS,
             'base_weights_sha256':base_sha, 'base_run':str(BASE_RUN),
             'tokenizer_sha256':header['tokenizer_signature']}
    del header
    phase_path.write_text(json.dumps(phase, indent=2))
if hashlib.sha256((OLD_DATA/'tokenizer.json').read_bytes()).hexdigest() != phase['tokenizer_sha256']:
    raise ValueError('Original tokenizer does not match checkpoint. Do not retrain BPE.')
print('Phase:', phase)


Phase: {'initial_step': 16149, 'initial_tokens': 30001152, 'additional_tokens': 20000000, 'target_tokens': 50001152, 'base_weights_sha256': 'c9171d3ad84e6593c81ef205f81f3c827925962d074e837972067e6b3273a4f5', 'base_run': '/content/drive/MyDrive/next-moe/checkpoint-30m-frozen', 'tokenizer_sha256': '9539edc24e68a0d143fc49ae5ea0b6d3dee9780b5930e0ce56a0210d7563f1c2'}


## Новий корпус: потокове читання, кеш і походження

Перший запуск займає час: progress друкується під час сканування. Цілі — training
**tokens**, не documents. Частки 80/20 наближені з точністю до останнього документа.
Використовуємо bounded shuffle, exact hashes і approximate SimHash дедуплікацію;
це не гарантія повної відсутності близьких дублікатів. Фільтруються короткі,
непрозові та повторювані тексти. Весь пул — природні джерельні тексти, classifiers
якості авторів мають LLM-анотації; PDF/OCR може містити помилки.

metadata.json містить dataset/config/revision, exact counts, reject counters,
мікс за tokens і hashes. documents.jsonl зберігає URL та походження кожного документа.
Після збою неповний каталог не використовується як готовий корпус. Помилки мережі
не підміняються toy data. У разі зміни corpus settings використовуйте новий RUN_NAME.


In [5]:
if (DATA_DIR/'metadata.json').exists():
    metadata = prepare_continuation_data(DATA_DIR, OLD_DATA,
        tokens_per_language=TOKENS_PER_LANGUAGE, validation_documents=NEW_VALIDATION_DOCUMENTS,
        pdf_fraction=PDF_FRACTION, seed=SEED, shuffle_buffer=SHUFFLE_BUFFER)
    shutil.copytree(DATA_DIR, CACHE_DIR, dirs_exist_ok=True)
else:
    build_dir = LOCAL_DIR/'corpus-build'
    metadata = prepare_continuation_data(build_dir, OLD_DATA,
        tokens_per_language=TOKENS_PER_LANGUAGE, validation_documents=NEW_VALIDATION_DOCUMENTS,
        pdf_fraction=PDF_FRACTION, seed=SEED, shuffle_buffer=SHUFFLE_BUFFER)
    estimated = sum(p.stat().st_size for p in build_dir.rglob('*') if p.is_file())
    used = sum(p.stat().st_size for p in RUN_DIR.rglob('*') if p.is_file())
    if (used+estimated+1.2*2**30)/2**30 > PERSISTENT_BUDGET_GIB:
        raise OSError('New run data plus two temporary weights snapshots exceed configured run budget.')
    pending = RUN_DIR/'data-stage3.tmp'
    if pending.exists():
        raise FileExistsError('Incomplete Drive corpus copy exists: rename/remove data-stage3.tmp before retry.')
    shutil.copytree(build_dir, pending)
    os.replace(pending, DATA_DIR)
    shutil.copytree(build_dir, CACHE_DIR, dirs_exist_ok=True)
print(json.dumps({k:metadata[k] for k in ('tokens','source_training_tokens','document_counts','rejected')}, indent=2))
tokenizer = Tokenizer.from_file(str(CACHE_DIR/'tokenizer.json'))
for lang in ('uk','en'):
    assert (CACHE_DIR/f'val_{lang}.bin').read_bytes() == (OLD_DATA/f'val_{lang}.bin').read_bytes()
print('Tokenizer reused; old validation byte-identical.')


Opening HuggingFaceFW/fineweb-2 ukr_Cyrl af9c13333eb981300149d5ca60a8e9d659b276b9


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Opening HuggingFaceFW/finepdfs-edu ukr_Cyrl 9cfabe2127faca99b3d5c4dc6d1fcb397399ebde


...

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Opening HuggingFaceFW/finepdfs-edu eng_Latn 9cfabe2127faca99b3d5c4dc6d1fcb397399ebde


Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

en scanned 500 train tokens 456512 / 20000000 rejected {'mixed_pdf_languages': 2, 'short_or_nonprose': 1}
en scanned 1000 train tokens 1010632 / 20000000 rejected {'mixed_pdf_languages': 8, 'short_or_nonprose': 3}
en scanned 1500 train tokens 1563942 / 20000000 rejected {'mixed_pdf_languages': 11, 'short_or_nonprose': 4, 'repeated_lines': 1}
en scanned 2000 train tokens 2166313 / 20000000 rejected {'mixed_pdf_languages': 12, 'short_or_nonprose': 5, 'repeated_lines': 1}
en scanned 2500 train tokens 2778150 / 20000000 rejected {'mixed_pdf_languages': 17, 'short_or_nonprose': 7, 'repeated_lines': 2}
en scanned 3000 train tokens 3373834 / 20000000 rejected {'mixed_pdf_languages': 23, 'short_or_nonprose': 7, 'repeated_lines': 2}
en scanned 3500 train tokens 3944351 / 20000000 rejected {'mixed_pdf_languages': 24, 'short_or_nonprose': 8, 'repeated_lines': 2}
en scanned 4000 train tokens 4506661 / 20000000 rejected {'mixed_pdf_languages': 27, 'short_or_nonprose': 8, 'repeated_lines': 2}
en sca

## Завантаження моделі та новий етап

З базового weights.pt — **warm restart**: ваги збережені, Adam новий, LR плавно
піднімається за 100 updates. LR ceiling 1e-4 для нового етапу (гіпотеза, стежте за validation).
Auto у поточному runtime обирає local last.pt з optimizer; після втрати runtime —
новий weights.pt на Drive. Читаємо model_config із checkpoint, не змінюємо архітектуру.
L4/BF16: weights і gradients моделі BF16, master weights/Adam FP32.


In [6]:
if RESUME_MODE == 'auto' and (LOCAL_DIR/'last.pt').exists():
    resume_path, mode = LOCAL_DIR/'last.pt', 'exact'
elif (RUN_DIR/'weights.pt').exists():
    resume_path, mode = RUN_DIR/'weights.pt', 'warm'
else:
    resume_path, mode = base_checkpoint, 'warm'
if mode == 'warm':
    local_weights = LOCAL_DIR/'input-weights.pt'
    shutil.copy2(resume_path, local_weights)
    resume_path = local_weights
header = torch.load(resume_path, map_location='cpu', weights_only=True, mmap=True)
model_cfg = ModelConfig(**header['model_config'])
if mode == 'exact':
    hash_path = RUN_DIR/'source_hashes.json'
    if hash_path.exists() and json.loads(hash_path.read_text()) != SOURCE_HASHES:
        raise ValueError('Exact resume code changed. Use original notebook or explicit RESUME_MODE=warm.')
    train_cfg = TrainConfig(**header['train_config'])
    if train_cfg.sequence_length != SEQ_LEN:
        raise ValueError('Exact resume seq_len differs; keep SEQ_LEN or explicitly use warm.')
else:
    precision = ('bf16_master' if torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
                 else 'fp16_master' if torch.cuda.is_available() else 'fp32') if PRECISION=='auto_master' else PRECISION
    per_step = SEQ_LEN*MICROBATCH*GRAD_ACCUM
    steps = header['step'] + math.ceil(max(phase['target_tokens']-header['tokens_seen'],0)/per_step)
    train_cfg = TrainConfig(steps=max(steps,1), sequence_length=SEQ_LEN, microbatch=MICROBATCH,
        accumulation=GRAD_ACCUM, learning_rate=LEARNING_RATE, warmup=1,
        restart_warmup=RESTART_WARMUP, precision=precision, seed=SEED,
        eval_batches=16, eval_every=500, save_every=500, log_every=10)
del header
trainer = Trainer(model_cfg, train_cfg, CACHE_DIR, RUN_DIR, resume=resume_path,
    resume_mode=mode, checkpoint_dir=LOCAL_DIR, persistent_weights=True,
    disk_budget_gib=PERSISTENT_BUDGET_GIB)
(RUN_DIR/'source_hashes.json').write_text(json.dumps(SOURCE_HASHES, indent=2))
fix_evaluation_microbatch(trainer, 1)
install_domain_evaluation(trainer)
install_phase_schedule(trainer, phase['initial_tokens'], phase['target_tokens'])
print('Resume:', mode, 'step:', trainer.step, 'tokens:', trainer.tokens_seen)
print('Goal:', phase['target_tokens'], 'remaining:', max(phase['target_tokens']-trainer.tokens_seen,0))
print('Initial validation:', trainer.evaluate())


WARM RESTART at step 16149: optimizer/RNG reset, 100-update LR ramp; not an exact continuation.
Parameters: 293,688,272; device=cuda; parameters=torch.bfloat16; autocast=torch.bfloat16; step=16149
Resume: warm step: 16149 tokens: 30001152
Goal: 50001152 remaining: 20000000
Initial validation: {'val_ce_uk': 3.989678392186761, 'val_ppl_uk': 54.037507683755365, 'val_ce_en': 5.111092448234558, 'val_ppl_en': 165.8514400555141, 'val_new_ce_uk': 6.299063175916672, 'val_new_ppl_uk': 544.0619809396848, 'val_new_ce_en': 5.847874224185944, 'val_new_ppl_en': 346.497022037946}


## Бенчмарк L4 і квота

Порівнюються microbatch 1/2/4 із незмінним effective batch (2048 tokens/update),
а за ввімкненого block checkpointing — також його вимкнення. Модель, Adam/RNG і
progress відновлюються після тестових updates. Benchmark витрачає час/квоту,
але не додає тестові tokens до навчання. Host RAM для backup ≈full checkpoint size.
На новому Adam спочатку виконується **один справжній** update. Тест автоматично
застосовує fastest layout. Якщо OOM, відновлює стан і зупиняється; зменшіть
MICROBATCH або ввімкніть checkpointing у **новій warm фазі**.

Введіть поточні L4 units/hour, щоб побачити tokens/unit. Старий G4 benchmark:
~6420 tokens/s при 8.9 units/hour ≈2.60M tokens/unit. L4 вигідніший, коли його
speed / units_per_hour > 6420 / 8.9 ≈721.3. Оцінка без validation/save overhead.


In [7]:
if RUN_EFFICIENCY_BENCHMARK and trainer.cfg.steps-trainer.step >= 13:
    if not trainer.optimizer.state:
        trainer.run(max_updates=1)
    benchmark_results = benchmark_batch_layout(trainer, units_per_hour=COMPUTE_UNITS_PER_HOUR)
    speed = max(r['median_training_tokens_per_second'] for r in benchmark_results)
    hours = max(phase['target_tokens']-trainer.tokens_seen,0)/speed/3600
    print('Estimated remaining TRAINING hours (exclude eval/save):', round(hours,2))
    if COMPUTE_UNITS_PER_HOUR:
        print('Estimated units:', round(hours*COMPUTE_UNITS_PER_HOUR,2))
else:
    print('Benchmark disabled or goal too close. Current batch:', trainer.cfg.microbatch, trainer.cfg.accumulation)


{"step": 16150, "train_ce": 5.846147537231445, "router_aux": 1.0513578653335571, "lr": 1.0000000000000002e-06, "tokens_seen": 30003200, "grad_norm": 2.425591468811035, "tokens_per_second_training_step": 1373.7519674669595, "data_loading_seconds": 0.000575573999867629, "unselected_expert_fraction_phase": 0.0, "max_expert_to_mean_load_phase": 2.734375, "unselected_expert_fraction_this_step": 0.0, "max_expert_to_mean_load": 2.734375, "peak_allocated_GiB": 5.200429439544678, "peak_reserved_GiB": 7.6875, "tokens_per_second_including_interval_overhead": 1372.1911531851101}
Saved /content/next-moe-local/next-moe-30m-plus20m-webpdf-v1/last.pt (3.83 GiB)
Saved /content/drive/MyDrive/next-moe/next-moe-30m-plus20m-webpdf-v1/weights.pt (0.55 GiB)
BENCHMARK: microbatch=1, accumulation=4, block_checkpointing=False; updates will be rolled back.
{"step": 16160, "train_ce": 6.066738128662109, "router_aux": 1.1031126081943512, "lr": 1.0999974386172087e-05, "tokens_seen": 30023680, "grad_norm": 2.7407193

## Навчання до +20M

SESSION_UPDATES=None означає весь залишок бюджету. Зупинка кнопкою Stop намагається
зберегти останній завершений optimizer step. Повторне виконання цієї ячейки
продовжує поточний trainer, не пересоздає Adam і не змінює LR budget.
Для графіків посеред навчання можна навмисно поставити SESSION_UPDATES=1000.


In [ ]:
started = time.perf_counter()
final_metrics = trainer.run(max_updates=SESSION_UPDATES)
print('Minutes:', round((time.perf_counter()-started)/60,2))
print('Step:', trainer.step, 'tokens_seen:', trainer.tokens_seen, 'phase goal:', phase['target_tokens'])
print('Validation:', final_metrics)
weights_path = RUN_DIR/'weights.pt'
print('Drive weights:', weights_path)
_ = (RUN_DIR/'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))


{"step": 16160, "train_ce": 6.066198348999023, "router_aux": 1.0463337898254395, "lr": 1.0999974386172087e-05, "tokens_seen": 30023680, "grad_norm": 2.7343921661376953, "tokens_per_second_training_step": 1953.3097620730819, "data_loading_seconds": 0.00041122600032394985, "unselected_expert_fraction_phase": 0.0, "max_expert_to_mean_load_phase": 1.7098721265792847, "unselected_expert_fraction_this_step": 0.0, "max_expert_to_mean_load": 2.1484375, "peak_allocated_GiB": 7.393598556518555, "peak_reserved_GiB": 8.08203125, "tokens_per_second_including_interval_overhead": 1933.504303105148}
{"step": 16170, "train_ce": 6.259652137756348, "router_aux": 1.0696094036102295, "lr": 2.099980440400199e-05, "tokens_seen": 30044160, "grad_norm": 2.4126651287078857, "tokens_per_second_training_step": 1925.5575755648053, "data_loading_seconds": 0.0005128469997544016, "unselected_expert_fraction_phase": 0.0, "max_expert_to_mean_load_phase": 1.9268972873687744, "unselected_expert_fraction_this_step": 0.0, 

## Графіки loss, routing і швидкості

Стара Wikipedia validation — суцільні лінії; new-domain holdout — пунктир.
Metrics цього етапу окремі від старого run. Final evaluation теж показується,
навіть якщо останній update не кратний eval_every. Router heatmap — поточна фаза.


In [ ]:
import matplotlib.pyplot as plt
records = [json.loads(line) for line in trainer.log_path.read_text().splitlines()]
fig, axes = plt.subplots(1,3,figsize=(17,4))
axes[0].plot([r['tokens_seen']/1e6 for r in records],[r['train_ce'] for r in records],alpha=.45,label='train')
for lang in ('uk','en'):
    for prefix, style in [('val_ce_', '-'), ('val_new_ce_', '--')]:
        key=prefix+lang; subset=[r for r in records if key in r]
        xs=[r['tokens_seen']/1e6 for r in subset]; ys=[r[key] for r in subset]
        if 'final_metrics' in globals() and key in final_metrics:
            xs.append(trainer.tokens_seen/1e6); ys.append(final_metrics[key])
        axes[0].plot(xs,ys,style,label=key)
axes[0].set(xlabel='Processed tokens (M)',ylabel='Cross entropy');axes[0].legend(fontsize=8)
counts=trainer.routing_total.float().cpu()
im=axes[1].imshow((counts/counts.mean(-1,keepdim=True).clamp_min(1)).numpy(),aspect='auto')
axes[1].set(xlabel='Expert',ylabel='Layer',title='Assignments / mean, phase');fig.colorbar(im,ax=axes[1])
axes[2].plot([r['tokens_seen']/1e6 for r in records],[r['tokens_per_second_training_step'] for r in records],alpha=.6,label='training step')
subset=[r for r in records if 'tokens_per_second_including_interval_overhead' in r]
axes[2].plot([r['tokens_seen']/1e6 for r in subset],[r['tokens_per_second_including_interval_overhead'] for r in subset],label='interval incl. overhead')
axes[2].set(xlabel='Processed tokens (M)',ylabel='Tokens / second');axes[2].legend(fontsize=8)
plt.tight_layout();plt.show()


## Перевірка генерації (base model, не чат)


In [ ]:
for prompt in ['Україна — це', 'The history of science']:
    torch.manual_seed(1234)
    ids=tokenizer.encode(prompt).ids
    with trainer.autocast():
        generated=trainer.model.generate(torch.tensor([ids],device=trainer.device),
            max_new_tokens=80, temperature=.8, top_k=40, context=SEQ_LEN,
            valid_vocab=tokenizer.get_vocab_size())
    print(tokenizer.decode(generated[0].tolist())); print()


## Збереження та наступне відкриття

Drive: RUN_DIR/weights.pt + tokenizer.json + data-stage3/ + phase.json + configs,
source_hashes.json, metrics.jsonl, pip-freeze.txt. Local: LOCAL_DIR/last.pt із Adam.
Не дублюємо full checkpoint на Drive автоматично. frozen checkpoint 30M залишається
окремо; загальна квота Drive включає обидва run та інші файли. Run limit 4.5GiB
не перевіряє квоту всього Google Drive. При відсутності місця save видасть помилку.

Документація: https://github.com/FactorialD/next-moe/blob/work/docs/continuation-colab.md
